# Untrained Qwen3-4B-Instruct-2507 baseline (Milestone 2 validation)

Runs `Qwen/Qwen3-4B-Instruct-2507` (revision `cdbee75f17c01a7cc42f958dc650907174af0554`), untrained, on the 800 validation cases: 550 tier 1 (splits v1 on dataset v2) and 250 tier 2 (splits v2 on dataset v3). The settings are fixed by `docs/milestone-2/EVAL_PROTOCOL.md`: fp16 on one T4, the model's chat template, greedy decoding, `max_new_tokens=128`, no repetition penalty, the shared prompt in `packages/hetren_baselines/prompt.py`.

**Setup:** Accelerator *GPU T4 x1*, Internet *on*. Add the private Kaggle Dataset holding `hetren-m2-bundle.tar.gz` (built with `python scripts/build_kaggle_bundle.py`). Use *Save Version → Save & Run All* so the run survives closing the browser.

**Order:** a 20-case fp16 smoke test (10 tier 1, 10 tier 2) runs first. If it fails, the notebook stops before the full run; bring the result back for a decision (no fallback is applied automatically).

In [ ]:
import glob, json, os, shutil, subprocess, tarfile

WORK = "/kaggle/working/hetren"
OUT = "/kaggle/working/qwen-untrained"

if not os.path.exists(WORK):
    unpacked = glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True)
    packed = glob.glob("/kaggle/input/**/hetren-m2-bundle.tar.gz", recursive=True)

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

print("bundle commit:", json.load(open(f"{WORK}/BUNDLE.json"))["commit"])
os.chdir(WORK)

In [ ]:
# Pinned versions (the repo's runtime requirements, plus transformers for Qwen3).
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## fp16 smoke test (20 cases)

In [ ]:
TIER1 = "--dataset artifacts/milestone-2/dataset-v2 --split docs/milestone-2/splits-v1/validation.txt"
TIER2 = "--dataset artifacts/milestone-2/dataset-v3 --split docs/milestone-2/splits-v2/validation.txt"

!python -m packages.support_evaluator.local_model_baseline {TIER1} --out {OUT} --smoke 10
!python -m packages.support_evaluator.local_model_baseline {TIER2} --out {OUT} --smoke 10

In [ ]:
reports = {tier: json.load(open(f"{OUT}/smoke-tier{tier}/smoke_report.json")) for tier in (1, 2)}
print(json.dumps(reports, indent=2))

if not all(r["fp16_ok"] for r in reports.values()):
    raise SystemExit("fp16 smoke test failed: stop here and report; do not change precision without approval")

## Full validation run (about 2 hours)

In [ ]:
!python -m packages.support_evaluator.local_model_baseline {TIER1} --out {OUT}
!python -m packages.support_evaluator.local_model_baseline {TIER2} --out {OUT}

In [ ]:
# Download qwen-untrained-results.tar.gz from the notebook's Output tab.
with tarfile.open("/kaggle/working/qwen-untrained-results.tar.gz", "w:gz") as results:
    results.add(OUT, arcname="qwen-untrained")

for tier in (1, 2):
    path = f"{OUT}/summary-tier{tier}.json"
    if os.path.exists(path):
        summary = json.load(open(path))
        print(tier, summary["decision_accuracy"], summary["failures"])